<a href="https://colab.research.google.com/github/ThinkingBeyond/BeyondAI-2025/blob/main/Tornike%20Khabeishvili%20and%20Mohamed%20Hassan/plotting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install numpy==2.0.2
!pip install matplotlib==3.10.0

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import glob
from color_palette import get_color_palette

In [ ]:
def load_multiple_runs(pickle_pattern):
    """
    Load multiple pickle files matching a pattern

    Args:
        pickle_pattern: glob pattern for pickle files (e.g., 'results_run*.pickle')

    Returns:
        list: List of results dictionaries
    """
    pickle_files = sorted(glob.glob(pickle_pattern))
    print(f"Found {len(pickle_files)} pickle files")

    all_runs = []
    for pf in pickle_files:
        with open(pf, 'rb') as f:
            data = pickle.load(f)
            all_runs.append(data)
            print(f"Loaded: {pf}")

    return all_runs

def aggregate_metrics(all_runs):
    """
    Aggregate metrics across multiple runs with mean and std

    Args:
        all_runs: list of results dictionaries

    Returns:
        dict: Aggregated results with mean, std, min, max for each solver
    """
    solver_names = list(all_runs[0].keys())

    aggregated = {}

    for solver_name in solver_names:
        # Initialize metric collectors for each solver
        metrics = {
            'compile_time': [],
            'train_time': [],
            'compile_gpu_peak_mb': [],
            'train_gpu_peak_mb': [],
            'train_gpu_peak_delta_mb': [],
            'train_cpu_memory_mb': [],
            'final_test_loss': [],
            'first_run_time': [],
            'losses': []  # Loss curves stored separately
        }

        # Collect metrics from all runs
        for run in all_runs:
            if solver_name in run:
                solver_data = run[solver_name]
                for key in metrics.keys():
                    if key == 'losses':
                        metrics[key].append(np.array(solver_data.get(key, [])))
                    else:
                        metrics[key].append(solver_data.get(key, np.nan))

        # Calculate statistics for scalar metrics
        stats = {}
        for key, values in metrics.items():
            if key == 'losses':
                continue  # Handle separately
            else:
                values = np.array(values)
                values = values[~np.isnan(values)]  # Filter out missing data
                if len(values) > 0:
                    stats[key] = {
                        'mean': np.mean(values),
                        'std': np.std(values),
                        'min': np.min(values),
                        'max': np.max(values),
                        'n': len(values)
                    }
                else:
                    stats[key] = {
                        'mean': np.nan,
                        'std': np.nan,
                        'min': np.nan,
                        'max': np.nan,
                        'n': 0
                    }

        # Handle loss curves with alignment
        if len(metrics['losses']) > 0:
            stats['losses'] = align_and_aggregate_losses(metrics['losses'])

        # Calculate convergence speed (loss at mid-training)
        convergence_losses = []
        for loss_curve in metrics['losses']:
            if len(loss_curve) > 0:
                target_epoch = min(50, len(loss_curve) // 2)
                if target_epoch > 0:
                    convergence_losses.append(loss_curve[target_epoch])

        if len(convergence_losses) > 0:
            stats['convergence_speed'] = {
                'mean': np.mean(convergence_losses),
                'std': np.std(convergence_losses),
                'n': len(convergence_losses)
            }
        else:
            stats['convergence_speed'] = {'mean': np.nan, 'std': np.nan, 'n': 0}

        # Calculate training stability (variance in log-loss)
        stability_values = []
        for loss_curve in metrics['losses']:
            if len(loss_curve) > 10:
                log_losses = np.log10(loss_curve[loss_curve > 0] + 1e-10)
                stability_values.append(np.std(log_losses))

        if len(stability_values) > 0:
            stats['training_stability'] = {
                'mean': np.mean(stability_values),
                'std': np.std(stability_values),
                'n': len(stability_values)
            }
        else:
            stats['training_stability'] = {'mean': np.nan, 'std': np.nan, 'n': 0}

        aggregated[solver_name] = stats

    return aggregated

def align_and_aggregate_losses(loss_arrays):
    """
    Align loss curves of different lengths and compute mean/std
    Handles variable-length training runs by padding with NaN

    Args:
        loss_arrays: list of numpy arrays with losses

    Returns:
        dict: mean, std, and individual curves
    """
    max_len = max(len(arr) for arr in loss_arrays)
    n_runs = len(loss_arrays)

    # Create aligned array with NaN padding
    aligned = np.full((n_runs, max_len), np.nan)
    for i, arr in enumerate(loss_arrays):
        aligned[i, :len(arr)] = arr

    # Calculate statistics ignoring NaN values
    mean_loss = np.nanmean(aligned, axis=0)
    std_loss = np.nanstd(aligned, axis=0)

    return {
        'mean': mean_loss,
        'std': std_loss,
        'individual': aligned
    }

def create_dual_metric_bar_chart(aggregated, output_dir='plots'):
    """
    Create bar chart with dual y-axes showing loss and time side-by-side
    Uses twin axes to accommodate different scales
    """
    Path(output_dir).mkdir(exist_ok=True)
    solver_names = list(aggregated.keys())

    # Extract metrics
    loss_means = [aggregated[s]['final_test_loss']['mean'] for s in solver_names]
    loss_stds = [aggregated[s]['final_test_loss']['std'] for s in solver_names]
    time_means = [aggregated[s]['train_time']['mean'] for s in solver_names]
    time_stds = [aggregated[s]['train_time']['std'] for s in solver_names]

    # Set color scheme
    bg_color = '#d9f4cd'
    grid_color = '#6B8E6F'
    text_color = '#2F3E30'

    plt.rcParams.update({
        'axes.facecolor': bg_color,
        'figure.facecolor': '#9ec193',
        'grid.color': grid_color,
        'grid.alpha': 0.25,
        'text.color': text_color,
        'axes.labelcolor': text_color,
        'xtick.color': text_color,
        'ytick.color': text_color,
        'font.size': 11,
        'axes.titlesize': 14,
        'axes.labelsize': 12
    })

    fig, ax1 = plt.subplots(figsize=(14, 7))

    x = np.arange(len(solver_names))
    width = 0.35

    color_loss = '#C07F66'  # Terra cotta
    color_time = '#5C7A5F'  # Sage green

    # Create second y-axis
    ax2 = ax1.twinx()

    # Plot bars on respective axes
    bars1 = ax1.bar(x - width/2, loss_means, width, yerr=loss_stds, capsize=5,
                    color=color_loss, alpha=0.9, edgecolor='white', linewidth=1,
                    error_kw={'elinewidth': 1.5, 'ecolor': '#2F3E30'}, label='Final Test Loss')

    bars2 = ax2.bar(x + width/2, time_means, width, yerr=time_stds, capsize=5,
                    color=color_time, alpha=0.9, edgecolor='white', linewidth=1,
                    error_kw={'elinewidth': 1.5, 'ecolor': '#2F3E30'}, label='Training Time (s)')

    # Configure axes
    ax1.set_xlabel('Solver', fontweight='bold')
    ax1.set_ylabel('Final Test Loss', fontweight='bold', color=color_time)
    ax2.set_ylabel('Total Training Time (s)', fontweight='bold', color=color_time)
    ax1.set_title('Final Test Loss and Total Training Time by Solver', fontweight='bold', pad=15)

    ax1.set_xticks(x)
    ax1.set_xticklabels(solver_names, rotation=45, ha='right')

    ax1.tick_params(axis='y', labelcolor=color_time)
    ax2.tick_params(axis='y', labelcolor=color_time)

    ax1.grid(True, axis='y', linestyle='--', linewidth=0.7, alpha=0.3)

    # Combine legends from both axes
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    legend = ax1.legend(lines1 + lines2, labels1 + labels2,
                       framealpha=0.95, edgecolor=grid_color, loc='upper left')
    legend.get_frame().set_facecolor(bg_color)

    plt.tight_layout()
    plt.savefig(f"{output_dir}/loss_and_time_comparison.png", dpi=300, bbox_inches='tight')
    plt.close()

    plt.rcParams.update(plt.rcParamsDefault)
    print(f"✓ Dual metric bar chart saved to {output_dir}/loss_and_time_comparison.png")

def plot_comparison_with_error_bars(aggregated, output_dir='plots'):
    """
    Generate comprehensive comparison plots:
    - Loss curves with multi-color palette
    - Bar charts with uniform sage green color
    """
    Path(output_dir).mkdir(exist_ok=True)
    solver_names = list(aggregated.keys())

    # Color scheme configuration
    bg_color = '#d9f4cd'
    grid_color = '#6B8E6F'
    text_color = '#2F3E30'

    loss_palette = get_color_palette(solver_names)  # Multi-color for loss curves
    uniform_bar_color = '#5C7A5F'  # Single color for bar charts

    plt.rcParams.update({
        'axes.facecolor': bg_color,
        'figure.facecolor': '#9ec193',
        'grid.color': grid_color,
        'grid.alpha': 0.25,
        'text.color': text_color,
        'axes.labelcolor': text_color,
        'xtick.color': text_color,
        'ytick.color': text_color,
        'font.size': 11,
        'axes.titlesize': 14,
        'axes.labelsize': 12
    })

    print(f"Generating plots in {output_dir}...")

    # Loss curves with multi-color palette
    plt.figure(figsize=(12, 7))

    for idx, solver_name in enumerate(solver_names):
        if 'losses' in aggregated[solver_name]:
            loss_data = aggregated[solver_name]['losses']
            mean_loss = loss_data['mean']
            std_loss = loss_data['std']
            epochs = np.arange(len(mean_loss))

            c = loss_palette[idx]

            # Plot mean line with shaded error region
            plt.plot(epochs, mean_loss, label=solver_name, linewidth=2.5, color=c, alpha=0.9)
            plt.fill_between(epochs, mean_loss - std_loss, mean_loss + std_loss,
                           alpha=0.2, color=c)

    plt.xlabel('Epoch', fontweight='bold')
    plt.ylabel('Loss', fontweight='bold')
    plt.yscale('log')
    plt.title('Training Loss Comparison (Mean ± Std)', fontweight='bold', pad=15)

    legend = plt.legend(framealpha=0.95, edgecolor=grid_color, loc='upper right')
    legend.get_frame().set_facecolor(bg_color)
    plt.grid(True, linestyle='--', linewidth=0.7)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/loss_comparison.png", dpi=300, bbox_inches='tight')
    plt.close()

    # Helper function for uniform bar charts
    def create_bar_chart(metric_key, title, filename, y_label):
        means = [aggregated[s][metric_key]['mean'] for s in solver_names]
        stds = [aggregated[s][metric_key]['std'] for s in solver_names]

        plt.figure(figsize=(12, 6))

        plt.bar(solver_names, means, yerr=stds, capsize=5,
                color=uniform_bar_color, alpha=0.9, edgecolor='white', linewidth=1,
                error_kw={'elinewidth': 1.5, 'ecolor': '#2F3E30'})

        plt.xlabel('Solver', fontweight='bold')
        plt.ylabel(y_label, fontweight='bold')
        plt.title(title, fontweight='bold', pad=15)
        plt.xticks(rotation=45, ha='right')
        plt.grid(True, axis='y', linestyle='--', linewidth=0.7)
        plt.tight_layout()
        plt.savefig(f"{output_dir}/{filename}", dpi=300, bbox_inches='tight')
        plt.close()

    # Generate standard bar charts
    create_bar_chart('train_time', 'Training Time Comparison', 'train_time.png', 'Time (s)')
    create_bar_chart('compile_time', 'Compilation Time Comparison', 'compile_time.png', 'Time (s)')
    create_bar_chart('train_gpu_peak_mb', 'Peak GPU Memory Usage', 'gpu_memory.png', 'Memory (MB)')
    create_bar_chart('final_test_loss', 'Final Test Loss', 'final_loss.png', 'Loss (MSE)')

    # Stacked time breakdown chart
    compile_means = [aggregated[s]['compile_time']['mean'] for s in solver_names]
    train_means = [aggregated[s]['train_time']['mean'] for s in solver_names]
    compile_stds = [aggregated[s]['compile_time']['std'] for s in solver_names]
    train_stds = [aggregated[s]['train_time']['std'] for s in solver_names]

    plt.figure(figsize=(12, 6))
    x = np.arange(len(solver_names))
    width = 0.65

    col_train = uniform_bar_color   # Dark sage
    col_compile = '#8FA892'         # Light sage

    # Stack compile and train time
    plt.bar(x, compile_means, width, label='Compilation',
            color=col_compile, alpha=0.95, edgecolor='white')
    plt.bar(x, train_means, width, bottom=compile_means, label='Training',
            color=col_train, alpha=0.95, edgecolor='white')

    # Add error bars for total time
    total_means = np.array(compile_means) + np.array(train_means)
    total_stds = np.sqrt(np.array(compile_stds)**2 + np.array(train_stds)**2)
    plt.errorbar(x, total_means, yerr=total_stds, fmt='none',
                 ecolor='#2F3E30', capsize=5, linewidth=1.5)

    plt.xlabel('Solver', fontweight='bold')
    plt.ylabel('Total Time (s)', fontweight='bold')
    plt.title('Time Breakdown: Compilation vs Training', fontweight='bold', pad=15)
    plt.xticks(x, solver_names, rotation=45, ha='right')

    legend = plt.legend(framealpha=0.95, edgecolor=grid_color)
    legend.get_frame().set_facecolor(bg_color)

    plt.grid(True, axis='y', linestyle='--', linewidth=0.7)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/time_breakdown.png", dpi=300, bbox_inches='tight')
    plt.close()

    plt.rcParams.update(plt.rcParamsDefault)
    print("✓ All plots generated successfully.")

def generate_latex_table(aggregated, output_file='results_table.tex'):
    """
    Generate compact LaTeX table with key metrics
    Includes compile time, train time, GPU memory, and final loss
    """
    solver_names = list(aggregated.keys())

    latex = []
    latex.append(r"\begin{table}[htbp]")
    latex.append(r"\centering")
    latex.append(r"\caption{Solver Performance Comparison (Mean $\pm$ Std)}")
    latex.append(r"\label{tab:solver_comparison}")
    latex.append(r"\begin{tabular}{l|rr|rr|r}")
    latex.append(r"\hline")
    latex.append(r"Solver & \multicolumn{2}{c|}{Time (s)} & \multicolumn{2}{c|}{GPU Memory (MB)} & Test Loss \\")
    latex.append(r" & Compile & Train & Peak (C) & Peak (T) & Final \\")
    latex.append(r"\hline")

    # Format helper function
    for solver_name in solver_names:
        data = aggregated[solver_name]

        def fmt(metric_name, precision=2):
            mean = data[metric_name]['mean']
            std = data[metric_name]['std']
            if np.isnan(mean):
                return "---"
            if np.isnan(std) or std == 0 or data[metric_name]['n'] <= 1:
                return f"{mean:.{precision}f}"
            return f"{mean:.{precision}f} $\\pm$ {std:.{precision}f}"

        safe_name = solver_name.replace('_', r'\_')

        row = (f"{safe_name} & "
               f"{fmt('compile_time')} & "
               f"{fmt('train_time')} & "
               f"{fmt('compile_gpu_peak_mb', 1)} & "
               f"{fmt('train_gpu_peak_mb', 1)} & "
               f"{fmt('final_test_loss', 6)} \\\\")

        latex.append(row)

    latex.append(r"\hline")
    latex.append(r"\end{tabular}")
    latex.append(r"\end{table}")

    with open(output_file, 'w') as f:
        f.write('\n'.join(latex))

    print(f"\n✓ LaTeX table saved to {output_file}")

    print("\n" + "="*80)
    print("LATEX TABLE")
    print("="*80)
    print('\n'.join(latex))
    print("="*80)

def generate_detailed_latex_table(aggregated, output_file='results_detailed_table.tex'):
    """
    Generate comprehensive LaTeX table with all available metrics
    Includes CPU memory and GPU delta in addition to standard metrics
    """
    solver_names = list(aggregated.keys())

    latex = []
    latex.append(r"\begin{table*}[htbp]")
    latex.append(r"\centering")
    latex.append(r"\caption{Detailed Solver Performance Metrics (Mean $\pm$ Std)}")
    latex.append(r"\label{tab:solver_detailed}")
    latex.append(r"\small")
    latex.append(r"\begin{tabular}{l|cc|cc|cc|c}")
    latex.append(r"\hline")
    latex.append(r"\multirow{2}{*}{Solver} & \multicolumn{2}{c|}{Time (s)} & \multicolumn{2}{c|}{GPU Peak (MB)} & \multicolumn{2}{c|}{Memory (MB)} & Test \\")
    latex.append(r" & Compile & Train & Compile & Train & CPU & GPU Delta & Loss \\")
    latex.append(r"\hline")

    for solver_name in solver_names:
        data = aggregated[solver_name]

        def fmt(metric_name, precision=2):
            mean = data[metric_name]['mean']
            std = data[metric_name]['std']
            if np.isnan(mean):
                return "---"
            if np.isnan(std) or std == 0 or data[metric_name]['n'] <= 1:
                return f"{mean:.{precision}f}"
            return f"{mean:.{precision}f} {{\\scriptsize $\\pm$ {std:.{precision}f}}}"

        safe_name = solver_name.replace('_', r'\_')

        row = (f"{safe_name} & "
               f"{fmt('compile_time')} & "
               f"{fmt('train_time')} & "
               f"{fmt('compile_gpu_peak_mb', 1)} & "
               f"{fmt('train_gpu_peak_mb', 1)} & "
               f"{fmt('train_cpu_memory_mb', 1)} & "
               f"{fmt('train_gpu_peak_delta_mb', 1)} & "
               f"{fmt('final_test_loss', 6)} \\\\")

        latex.append(row)

    latex.append(r"\hline")
    latex.append(r"\end{tabular}")
    latex.append(r"\end{table*}")

    with open(output_file, 'w') as f:
        f.write('\n'.join(latex))

    print(f"\n✓ Detailed LaTeX table saved to {output_file}")

def plot_loss_vs_compute_time(aggregated, output_dir='plots'):
    """
    Plot loss curves against wall-clock time instead of epochs
    Scales each algorithm's epochs by their average time per epoch
    """
    Path(output_dir).mkdir(exist_ok=True)
    solver_names = list(aggregated.keys())

    bg_color = '#d9f4cd'
    grid_color = '#6B8E6F'
    text_color = '#2F3E30'

    loss_palette = get_color_palette(solver_names)

    plt.rcParams.update({
        'axes.facecolor': bg_color,
        'figure.facecolor': '#9ec193',
        'grid.color': grid_color,
        'grid.alpha': 0.25,
        'text.color': text_color,
        'axes.labelcolor': text_color,
        'xtick.color': text_color,
        'ytick.color': text_color,
        'font.size': 11,
        'axes.titlesize': 14,
        'axes.labelsize': 12
    })

    plt.figure(figsize=(12, 7))

    for idx, solver_name in enumerate(solver_names):
        if 'losses' in aggregated[solver_name]:
            loss_data = aggregated[solver_name]['losses']
            mean_loss = loss_data['mean']
            std_loss = loss_data['std']

            # Calculate time per epoch
            total_train_time = aggregated[solver_name]['train_time']['mean']
            num_epochs = len(mean_loss)
            time_per_epoch = total_train_time / num_epochs

            # Create cumulative time array
            compute_times = np.arange(num_epochs) * time_per_epoch

            c = loss_palette[idx]

            plt.plot(compute_times, mean_loss, label=solver_name,
                    linewidth=2.5, color=c, alpha=0.9)
            plt.fill_between(compute_times,
                           mean_loss - std_loss,
                           mean_loss + std_loss,
                           alpha=0.2, color=c)

    plt.xlabel('Compute Time (seconds)', fontweight='bold')
    plt.ylabel('Loss', fontweight='bold')
    plt.yscale('log')
    plt.title('Training Loss vs Compute Time (Mean ± Std)', fontweight='bold', pad=15)

    legend = plt.legend(framealpha=0.95, edgecolor=grid_color, loc='upper right')
    legend.get_frame().set_facecolor(bg_color)
    plt.grid(True, linestyle='--', linewidth=0.7)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/loss_vs_compute_time.png", dpi=300, bbox_inches='tight')
    plt.close()

    plt.rcParams.update(plt.rcParamsDefault)
    print(f"✓ Loss vs Compute Time plot saved to {output_dir}/loss_vs_compute_time.png")

def plot_parallel_coordinates(aggregated, output_dir='plots'):
    """
    Create parallel coordinates plot showing normalized performance across metrics
    Higher values indicate better performance for all metrics
    """
    Path(output_dir).mkdir(exist_ok=True)

    # Normalize metrics to 0-1 scale
    normalized = normalize_metrics(aggregated)
    solver_names = list(normalized.keys())

    # Define metrics and labels for plot
    metrics = ['final_test_loss', 'train_time', 'train_gpu_peak_mb',
               'convergence_speed', 'training_stability']
    metric_labels = ['Final\nAccuracy', 'Training\nSpeed', 'Memory\nEfficiency',
                     'Early\nConvergence', 'Training\nStability']

    bg_outer = '#9ec193'
    bg_inner = '#d9f4cd'

    solver_colors = get_color_palette(solver_names)
    print("Solver names:", solver_names)
    print("Colors assigned:", solver_colors)

    fig, ax = plt.subplots(figsize=(14, 8))

    fig.patch.set_facecolor(bg_outer)
    ax.set_facecolor(bg_inner)

    x_positions = np.arange(len(metrics))

    # Plot each solver as a connected line
    for idx, solver in enumerate(solver_names):
        color = solver_colors[idx]
        values = [normalized[solver][m] for m in metrics]

        ax.plot(x_positions, values,
                marker='o', linewidth=2.5, markersize=8,
                color=color, alpha=0.85, label=solver)

    # Configure plot
    ax.set_xticks(x_positions)
    ax.set_xticklabels(metric_labels, fontsize=12, fontweight='bold', color='#000000')
    ax.set_ylabel('Normalized Performance (Higher = Better)',
                  fontsize=13, fontweight='bold', color='#000000')
    ax.set_ylim(-0.05, 1.1)
    ax.set_xlim(-0.2, len(metrics) - 0.8)

    ax.grid(True, axis='y', linestyle='--', linewidth=0.7, alpha=1, color='black')
    ax.set_axisbelow(True)

    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(2)

    # Add vertical reference lines
    for x in x_positions:
        ax.axvline(x, color='gray', linewidth=0.5, alpha=0.3)

    ax.set_title('SDE Solver Performance: Parallel Coordinates',
                 fontsize=16, fontweight='bold', pad=20)

    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
              framealpha=0.95, fontsize=11)

    plt.tight_layout()
    plt.savefig(f"{output_dir}/parallel_coordinates.png",
                dpi=300, bbox_inches='tight')
    plt.close()

    print(f"✓ Parallel coordinates plot saved to {output_dir}/parallel_coordinates.png")

def normalize_metrics(aggregated):
    """
    Normalize metrics to 0-1 scale based on best performer
    Best performer (minimum value) gets 1.0, others scaled proportionally

    Args:
        aggregated: dict with aggregated metrics

    Returns:
        dict: normalized metrics where higher is better
    """
    solver_names = list(aggregated.keys())
    metrics = ['final_test_loss', 'train_time', 'train_gpu_peak_mb',
               'convergence_speed', 'training_stability']

    # Collect all values to find best performer
    metric_values = {m: [] for m in metrics}
    for solver in solver_names:
        for metric in metrics:
            val = aggregated[solver][metric]['mean']
            if not np.isnan(val):
                metric_values[metric].append(val)

    # Find best (minimum) value for each metric
    best_values = {}
    for metric in metrics:
        if len(metric_values[metric]) > 0:
            best_values[metric] = min(metric_values[metric])
        else:
            best_values[metric] = 1.0

    # Normalize: best=1.0, worse=proportionally lower
    normalized = {}
    for solver in solver_names:
        normalized[solver] = {}
        for metric in metrics:
            val = aggregated[solver][metric]['mean']
            if np.isnan(val):
                normalized[solver][metric] = 0
            else:
                best_val = best_values[metric]

                if best_val == 0:
                    best_val = 1e-10

                relative_performance = best_val / val
                normalized[solver][metric] = min(1.0, relative_performance)

    return normalized

In [ ]:
if __name__ == "__main__":
    # Load all pickle files matching pattern
    pickle_pattern = "*.pickle"

    all_runs = load_multiple_runs(pickle_pattern)

    if len(all_runs) == 0:
        print("No pickle files found! Check your pattern.")
    else:
        # Aggregate across runs
        aggregated = aggregate_metrics(all_runs)

        # Generate all plots
        plot_comparison_with_error_bars(aggregated, output_dir='plots_with_error')
        create_dual_metric_bar_chart(aggregated, output_dir='plots_with_error')
        plot_loss_vs_compute_time(aggregated, output_dir='plots_with_error')
        plot_parallel_coordinates(aggregated, output_dir='plots_parallel')

        # Generate LaTeX tables
        generate_latex_table(aggregated, 'results_table.tex')
        generate_detailed_latex_table(aggregated, 'results_detailed_table.tex')

        print("\n✓ All done!")